In [2]:
import os
import sys

import boto3
from botocore.exceptions import BotoCoreError, ClientError


BUCKET = "cct-ds-code-challenge-input-data"
REGION = "af-south-1"
OBJECT_KEYS = (
    "city-hex-polygons-8-10.geojson",
    "city-hex-polygons-8.geojson",
)
PREVIEW_BYTES = 2048

In [3]:
def require_environment(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Required environment variable is missing: {name}")
    return value


def preview_object(s3_client, object_key: str) -> None:
    metadata = s3_client.head_object(Bucket=BUCKET, Key=object_key)
    print(
        f"{object_key}: accessible; "
        f"size={metadata.get('ContentLength')}; "
        f"content_type={metadata.get('ContentType')}; "
        f"etag={metadata.get('ETag')}"
    )

    response = s3_client.get_object(
        Bucket=BUCKET,
        Key=object_key,
        Range=f"bytes=0-{PREVIEW_BYTES - 1}",
    )
    preview = response["Body"].read(PREVIEW_BYTES).decode("utf-8", errors="replace")
    print(f"{object_key}: preview={preview[:200]!r}")





In [ ]:


# access_key = require_environment("AWS_ACCESS_KEY_ID")
# secret_key = require_environment("AWS_SECRET_ACCESS_KEY")

s3 = boto3.client(
    "s3",
    region_name=REGION,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
)

for object_key in OBJECT_KEYS:
    preview_object(s3_client, object_key)

print("S3 connectivity check passed")


city-hex-polygons-8-10.geojson: accessible; size=108254980; content_type=application/geo+json; etag="1b257eed30f40a4645b4021567fcf0b7-7"
city-hex-polygons-8-10.geojson: preview='{\n"type": "FeatureCollection",\n"crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:OGC:1.3:CRS84" } },\n"features": [\n{ "type": "Feature", "properties": { "index": "88ad361801fffff", "cen'
city-hex-polygons-8.geojson: accessible; size=1966685; content_type=application/geo+json; etag="88f619d038fda0dc7d7260bde74ad9ff"
city-hex-polygons-8.geojson: preview='{\n"type": "FeatureCollection",\n"name": "city-hex-polygons-8",\n"crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:OGC:1.3:CRS84" } },\n"features": [\n{ "type": "Feature", "properties": { "'
S3 connectivity check passed


In [10]:
import json
import pandas as pd

response = s3.get_object(
    Bucket=BUCKET,
    Key="city-hex-polygons-8-10.geojson",
)

geojson_data = json.loads(
    response["Body"].read().decode("utf-8")
)

geojson_data["features"] = [
    feature
    for feature in geojson_data["features"]
    if feature["properties"]["resolution"] == 8
]

df = pd.json_normalize(
    geojson_data["features"],
    sep="_",
)

df.head()

,type,properties_index,properties_centroid_lat,properties_centroid_lon,properties_resolution,geometry_type,geometry_coordinates
0,Feature,88ad361801fffff,-33.859427,18.677843,8,Polygon,"[[[18.6811898997334, -33.86330279081797], [18...."
1,Feature,88ad361803fffff,-33.855696,18.668766,8,Polygon,"[[[18.672112346191998, -33.85957172360946], [1..."
2,Feature,88ad361805fffff,-33.855263,18.685959,8,Polygon,"[[[18.68930552859897, -33.859138049118094], [1..."
3,Feature,88ad361807fffff,-33.851532,18.676881,8,Polygon,"[[[18.68022760998973, -33.85540739558428], [18..."
4,Feature,88ad361809fffff,-33.867322,18.678806,8,Polygon,"[[[18.682152273791363, -33.871197410257686], [..."


In [11]:
df.shape

(3832, 7)

In [ ]:
# Download the validation GeoJSON
response = s3.get_object(
    Bucket=BUCKET,
    Key="city-hex-polygons-8.geojson",
)

validation_geojson = json.loads(
    response["Body"].read().decode("utf-8")
)

validation_features = validation_geojson["features"]

print(f"Validation features: {len(validation_features):,}")
print(validation_features[0]["properties"])

Validation features: 3,832
{'index': '88ad361801fffff', 'centroid_lat': -33.859427322761434, 'centroid_lon': 18.677843311941835}


In [16]:
validation_df = pd.json_normalize(
    validation_features,
    sep="_",
)

validation_df.head()

,type,properties_index,properties_centroid_lat,properties_centroid_lon,geometry_type,geometry_coordinates
0,Feature,88ad361801fffff,-33.859427,18.677843,Polygon,"[[[18.6811898997334, -33.86330279081797], [18...."
1,Feature,88ad361803fffff,-33.855696,18.668766,Polygon,"[[[18.672112346191998, -33.85957172360946], [1..."
2,Feature,88ad361805fffff,-33.855263,18.685959,Polygon,"[[[18.68930552859897, -33.859138049118094], [1..."
3,Feature,88ad361807fffff,-33.851532,18.676881,Polygon,"[[[18.68022760998973, -33.85540739558428], [18..."
4,Feature,88ad361809fffff,-33.867322,18.678806,Polygon,"[[[18.682152273791363, -33.871197410257686], [..."


In [14]:
import json
import time

import numpy as np
import pandas as pd

In [22]:
import time


def validate_resolution_8(original_df, validation_df):
    start_time = time.perf_counter()

    index_column = "properties_index"

    original_indexes = set(
        original_df[index_column].dropna().astype(str)
    )

    validation_indexes = set(
        validation_df[index_column].dropna().astype(str)
    )

    matching_indexes = (
        original_indexes.intersection(validation_indexes)
    )

    missing_indexes = validation_indexes - original_indexes
    unexpected_indexes = original_indexes - validation_indexes

    volume_score = min(
        len(original_df),
        len(validation_df),
    ) / max(len(validation_df), 1)

    coverage_score = (
        len(matching_indexes)
        / max(len(validation_indexes), 1)
    )

    uniqueness_score = float(
        original_df[index_column].is_unique
        and validation_df[index_column].is_unique
    )

    score = (
        volume_score * 0.25
        + coverage_score * 0.50
        + uniqueness_score * 0.25
    ) * 100

    result = {
        "score": round(score, 2),
        "passed": score >= 99.5,
        "original_row_count": len(original_df),
        "validation_row_count": len(validation_df),
        "matching_index_count": len(matching_indexes),
        "missing_index_count": len(missing_indexes),
        "unexpected_index_count": len(unexpected_indexes),
        "original_indexes_unique": original_df[index_column].is_unique,
        "validation_indexes_unique": validation_df[index_column].is_unique,
        "missing_indexes": missing_indexes,
        "unexpected_indexes": unexpected_indexes,
        "elapsed_seconds": round(
            time.perf_counter() - start_time,
            4,
        ),
    }

    print(f"Original rows: {result['original_row_count']:,}")
    print(f"Validation rows: {result['validation_row_count']:,}")
    print(f"Matching indexes: {result['matching_index_count']:,}")
    print(f"Missing indexes: {result['missing_index_count']:,}")
    print(f"Unexpected indexes: {result['unexpected_index_count']:,}")
    print(f"Score: {result['score']}/100")
    print(f"Passed: {result['passed']}")
    print(f"Time: {result['elapsed_seconds']} seconds")

    return result

In [23]:
validation_result = validate_resolution_8(
    original_df=df,
    validation_df=validation_df,
)

Original rows: 3,832
Validation rows: 3,832
Matching indexes: 3,832
Missing indexes: 0
Unexpected indexes: 0
Score: 100.0/100
Passed: True
Time: 0.0226 seconds


In [24]:
import io
import json
from pathlib import Path

import boto3
import pandas as pd
import yaml


def validate_resolution_8_extraction(
    s3_client,
    bucket,
    contract_path="../config/data_extraction_contract.yml",
):
    # Load validation rules
    with Path(contract_path).open() as file:
        contract = yaml.safe_load(file)

    index_column = contract["comparison"]["index_column"]
    expected_resolution = contract["comparison"]["expected_resolution"]
    minimum_score = contract["scoring"]["pass_threshold"]
    expected_columns = set(contract["expected_shared_columns"])

    # Load source GeoJSON
    source_response = s3_client.get_object(
        Bucket=bucket,
        Key="city-hex-polygons-8-10.geojson",
    )

    source_geojson = json.loads(
        source_response["Body"].read().decode("utf-8")
    )

    # Keep only resolution-8 features
    source_features = [
        feature
        for feature in source_geojson["features"]
        if feature["properties"].get("resolution") == expected_resolution
    ]

    extracted_df = pd.json_normalize(source_features, sep="_")

    # Load reference GeoJSON
    validation_response = s3_client.get_object(
        Bucket=bucket,
        Key="city-hex-polygons-8.geojson",
    )

    validation_geojson = json.loads(
        validation_response["Body"].read().decode("utf-8")
    )

    validation_df = pd.json_normalize(
        validation_geojson["features"],
        sep="_",
    )

    # Check required shared columns
    shared_columns_passed = (
        expected_columns.issubset(extracted_df.columns)
        and expected_columns.issubset(validation_df.columns)
    )

    # Check row counts
    row_count_score = min(
        len(extracted_df),
        len(validation_df),
    ) / max(len(validation_df), 1)

    # Compare H3 indexes
    extracted_indexes = set(
        extracted_df[index_column].dropna().astype(str)
    )

    validation_indexes = set(
        validation_df[index_column].dropna().astype(str)
    )

    matching_indexes = (
        extracted_indexes.intersection(validation_indexes)
    )

    index_coverage_score = (
        len(matching_indexes)
        / max(len(validation_indexes), 1)
    )

    # Check uniqueness
    index_uniqueness_passed = (
        extracted_df[index_column].is_unique
        and validation_df[index_column].is_unique
    )

    index_uniqueness_score = float(index_uniqueness_passed)

    # Calculate weighted score from the YAML contract
    weights = {
        name: details["weight"]
        for name, details in contract["checks"].items()
        if details.get("enabled", False)
    }

    score_components = {
        "shared_columns": float(shared_columns_passed),
        "row_count": row_count_score,
        "index_coverage": index_coverage_score,
        "index_uniqueness": index_uniqueness_score,
    }

    score = sum(
        score_components[name] * weights[name]
        for name in weights
    ) * 100

    passed = score >= minimum_score

    print(f"Extracted rows: {len(extracted_df):,}")
    print(f"Validation rows: {len(validation_df):,}")
    print(f"Matching indexes: {len(matching_indexes):,}")
    print(f"Score: {score:.2f}/100")
    print(f"Passed: {passed}")

    return passed

In [25]:
passed = validate_resolution_8_extraction(
    s3_client=s3,
    bucket="cct-ds-code-challenge-input-data",
)

passed

Extracted rows: 3,832
Validation rows: 3,832
Matching indexes: 3,832
Score: 100.00/100
Passed: True


True

In [4]:
import json
import time
from pathlib import Path
import boto3
import pandas as pd
import yaml


def validate_extraction_with_s3_select(
    s3_client,
    bucket,
    contract_path="../config/data_extraction_contract.yml",
):
    start_time = time.perf_counter()

    # Read the contract
    with Path(contract_path).open() as file:
        contract = yaml.safe_load(file)

    index_column = contract["comparison"]["index_column"]
    resolution = contract["comparison"]["expected_resolution"]
    threshold = contract["scoring"]["pass_threshold"]
    scale = contract["scoring"]["scale"]

    weights = {
        name: check["weight"]
        for name, check in contract["checks"].items()
        if check["enabled"]
    }

    expected_columns = set(
        contract["expected_shared_columns"]
    )

    # Extract resolution-8 features using S3 Select
    query = f"""
        SELECT feature
        FROM S3Object[*].features[*] AS feature
        WHERE feature.properties.resolution = {resolution}
    """

    response = s3_client.select_object_content(
        Bucket=bucket,
        Key="city-hex-polygons-8-10.geojson",
        ExpressionType="SQL",
        Expression=query,
        InputSerialization={
            "JSON": {
                "Type": "DOCUMENT"
            }
        },
        OutputSerialization={
            "JSON": {
                "RecordDelimiter": "\n"
            }
        },
    )

    records = []
    buffer = ""

    for event in response["Payload"]:
        if "Records" not in event:
            continue

        buffer += event["Records"]["Payload"].decode("utf-8")
        lines = buffer.split("\n")
        buffer = lines.pop()

        for line in lines:
            if line.strip():
                records.append(json.loads(line)["feature"])

    if buffer.strip():
        records.append(json.loads(buffer)["feature"])

    extracted_df = pd.json_normalize(records, sep="_")

    # Load the reference dataset
    reference_response = s3_client.get_object(
        Bucket=bucket,
        Key="city-hex-polygons-8.geojson",
    )

    reference_geojson = json.loads(
        reference_response["Body"].read().decode("utf-8")
    )

    reference_df = pd.json_normalize(
        reference_geojson["features"],
        sep="_",
    )

    # Check required shared columns
    shared_columns_score = float(
        expected_columns.issubset(extracted_df.columns)
        and expected_columns.issubset(reference_df.columns)
    )

    # Check row counts
    row_count_score = min(
        len(extracted_df),
        len(reference_df),
    ) / max(len(reference_df), 1)

    # Check index coverage
    extracted_indexes = set(
        extracted_df[index_column].dropna().astype(str)
    )

    reference_indexes = set(
        reference_df[index_column].dropna().astype(str)
    )

    index_coverage_score = (
        len(extracted_indexes & reference_indexes)
        / max(len(reference_indexes), 1)
    )

    # Check index uniqueness
    index_uniqueness_score = float(
        extracted_df[index_column].is_unique
        and reference_df[index_column].is_unique
    )

    score_components = {
        "shared_columns": shared_columns_score,
        "row_count": row_count_score,
        "index_coverage": index_coverage_score,
        "index_uniqueness": index_uniqueness_score,
    }

    score = sum(
        score_components[name] * weights[name]
        for name in weights
    ) * scale

    elapsed_seconds = time.perf_counter() - start_time
    passed = score >= threshold

    print(f"Time: {elapsed_seconds:.4f} seconds")
    print(f"Score: {score:.2f}/100")
    print(f"Passed: {passed}")

    return {
        "score": round(score, 2),
        "passed": passed,
        "elapsed_seconds": round(elapsed_seconds, 4),
    }

In [ ]:

BUCKET = "cct-ds-code-challenge-input-data"
REGION = "af-south-1"
OBJECT_KEYS = (
    "city-hex-polygons-8-10.geojson",
    "city-hex-polygons-8.geojson",
)
PREVIEW_BYTES = 2048

# access_key = require_environment("AWS_ACCESS_KEY_ID")
# secret_key = require_environment("AWS_SECRET_ACCESS_KEY")

s3 = boto3.client(
    "s3",
    region_name=REGION,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
)

validation_result = validate_extraction_with_s3_select(
    s3_client=s3,
    bucket=BUCKET,
)

Time: 3.8197 seconds
Score: 100.00/100
Passed: True


In [7]:
from data_extraction import DataExtraction

extraction = DataExtraction(
    s3_client=s3,
    bucket=BUCKET,
)

result = extraction.run()

result

{'time_seconds': 3.7991, 'score': 100.0, 'passed': True}

In [13]:
CREDENTIALS_URL = (
    "https://cct-ds-code-challenge-input-data.s3.af-south-1."
    "amazonaws.com/ds_code_challenge_creds.json"
)

BUCKET = "cct-ds-code-challenge-input-data"
REGION = "af-south-1"


import json
import ssl
import urllib.request

import certifi


def load_credentials():
    ssl_context = ssl.create_default_context(
        cafile=certifi.where()
    )

    with urllib.request.urlopen(
        CREDENTIALS_URL,
        context=ssl_context,
    ) as response:
        credentials = json.load(response)

    return credentials["s3"]

In [14]:
credentials = load_credentials()